# gatenet colab-v3-rot -- camera-roll augmentation

Same machinery as `gatenet_colab_v3.ipynb`: imports `gatenet.py`, rebuilds the crop cache
with `packcrops.py`, calls `gatenet.train()`. The baseline this run is measured against is
**`colab-v3-nw`** (the current production per TRAINING.md's 2026-08-01 audit: v3 merged
labels, NO rate weighting). Exactly ONE thing changes versus that baseline:

* **rotation (camera-roll) augmentation on the TRAIN loader only** (`gatenet_rotaug.py`).
  VQ1 footage is slow, level flying; the VQ2 race is acro with large bank, so the net has
  never seen a rolled gate. For a pinhole camera a pure roll is an EXACT homography
  H = K R K^-1 (K: fx=fy=320, cx=cy=320,180), so warping tile and corner labels by the SAME
  affine keeps the pair geometrically consistent -- no new label noise.

Everything else is held at the `colab-v3-nw` baseline: **labels `labels_merged_v3.json`,
block split, weight-mode `none`, batch 256, lr 3e-4*(batch/64), 200 epochs, seed 0**. The
val loader is the untouched `CachedGateCrops`, so the val set is identical to `colab-v3-nw`'s
and the cross-eval is a like-for-like comparison with NO home-field leakage (both models
trained on the same label file => same block boundaries => same held-out frames).

**Roll is tile-level only, capped at the safe angle.** `gatenet_rotaug.safe_angle_deg()`
computes it from the cache constants: tiles are packed at TILE_SCALE=2.10 but the crop is
MARGIN=1.60 of the visible box, so a roll of A about the TILE centre keeps the crop inside
the tile while cos A + sin A <= 2.10/1.60 = 1.3125, i.e. **A = 23.14 deg**. Rolling about the
tile centre (not the jittered crop centre) keeps the aperture near the pivot; verified on the
laptop (`rotaug_check.py`) that this never pulls beyond-tile black onto an unclipped aperture.
Full-frame roll (which also translates off-centre gates) is deliberately OUT of scope here --
one variable.

---

## WHAT TO UPLOAD, AND WHERE

No local pre-step beyond the ones the v3 run already did (`mergelabels.py`,
`_mkframezip_vq2.py`). On Drive, in **`MyDrive/vqual2/`**:

```
MyDrive/vqual2/
  perception/
    gatenet.py               <- CURRENT copy
    packcrops.py             <- CURRENT copy
    gatenet_rotaug.py        <- NEW (this run's only new code)
    autolabel.py             <- unchanged (canon)
    labels_merged_v3.json    <- the v3 merge (same file colab-v3-nw used)
    val_clean_keys.json      <- for the clean-subset cross-check
  vq1_frames.zip             <- already there
  vq2_frames.zip             <- already there (from the v3 run)
  gatenet_runs/
    block/best.pt            <- already there (OLD production, clean-subset only)
    colab-v3-nw/best.pt      <- already there; the DIRECT baseline this run must beat
```

`gatenet_rotaug.py` is the one file to drag up that the v3 run did not need. Everything
else is already on Drive from the v3 upload. Upload zips with the Drive client, not a cell.

## 1. What did Colab actually give us?

Colab varies by the hour: T4 / L4 / A100. Batch, workers and whether the cache fits RAM
depend on it, so measure rather than assume.

In [ ]:
import os, subprocess, sys, time, shutil, json

print("--- GPU " + "-" * 60)
try:
    print(subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
         "--format=csv,noheader"], text=True).strip())
except Exception as e:
    print("no nvidia-smi:", e)

import torch
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"  {p.name}  {p.total_memory/2**30:.1f} GiB  sm_{p.major}{p.minor}  "
          f"{p.multi_processor_count} SMs")

print("--- CPU / RAM " + "-" * 53)
print("vCPU (os.cpu_count):", os.cpu_count())
mem = {}
for line in open("/proc/meminfo"):
    k, v = line.split(":", 1)
    mem[k] = v.strip()
print("MemTotal:", mem.get("MemTotal"), "| MemAvailable:", mem.get("MemAvailable"))
print("--- DISK (/content) " + "-" * 48)
t, u, f = shutil.disk_usage("/content")
print(f"total {t/2**30:.0f} GiB, free {f/2**30:.0f} GiB")
assert f > 8 * 2**30, "not enough local disk for the ~4.5 GB cache + checkpoints"

## 2. Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE       = "/content/drive/MyDrive/vqual2"   # <- the one path to change
CODE_DRIVE  = f"{DRIVE}/perception"
LABELS_NAME = "labels_merged_v3.json"           # same label file colab-v3-nw used
VQ1_ZIP     = f"{DRIVE}/vq1_frames.zip"
VQ2_ZIP     = f"{DRIVE}/vq2_frames.zip"
RUNS_DRIVE  = f"{DRIVE}/gatenet_runs"
NW_BEST     = f"{RUNS_DRIVE}/colab-v3-nw/best.pt"  # DIRECT baseline (same labels, same split)
OLD_BEST    = f"{RUNS_DRIVE}/block/best.pt"        # OLD production, clean-subset cross-check

CODE_LOCAL  = "/content/code"
SESS_LOCAL  = "/content/sessions"
LOCAL_CACHE = "/content/crop_cache"

for p_ in (DRIVE, CODE_DRIVE):
    assert os.path.isdir(p_), f"missing on Drive: {p_}  (see the upload list at the top)"
for f_ in (VQ1_ZIP, VQ2_ZIP, f"{CODE_DRIVE}/{LABELS_NAME}",
           f"{CODE_DRIVE}/gatenet_rotaug.py", f"{CODE_DRIVE}/val_clean_keys.json"):
    assert os.path.isfile(f_), f"missing on Drive: {f_}  (see the upload list at the top)"
# checked NOW, not after 3 h: the whole point of this run is the comparison
assert os.path.isfile(NW_BEST), f"missing: {NW_BEST} -- the colab-v3-nw baseline is required"
if not os.path.isfile(OLD_BEST):
    print(f"NOTE: {OLD_BEST} absent -- the clean-subset 3-way will skip the OLD column")
os.makedirs(RUNS_DRIVE, exist_ok=True)
print("Drive OK.", LABELS_NAME,
      f"{os.path.getsize(f'{CODE_DRIVE}/{LABELS_NAME}')/1e6:.1f} MB;",
      f"zips {os.path.getsize(VQ1_ZIP)/1e6:.0f} + {os.path.getsize(VQ2_ZIP)/1e6:.0f} MB")

## 3. Unzip the frames and REBUILD the cache here

Identical to the v3 notebook: upload the small thing (frames), rebuild the big thing (tiles)
on local NVMe against `labels_merged_v3.json`, which becomes the cache's staleness
fingerprint (`labels_sha256`). `missing_frames` MUST be 0.

In [ ]:
import zipfile

os.makedirs(f"{CODE_LOCAL}/pilot", exist_ok=True)
shutil.rmtree(f"{CODE_LOCAL}/pilot/perception", ignore_errors=True)
shutil.copytree(CODE_DRIVE, f"{CODE_LOCAL}/pilot/perception")
LABELS_LOCAL = f"{CODE_LOCAL}/pilot/perception/{LABELS_NAME}"

if not os.path.isdir(SESS_LOCAL):
    os.makedirs(SESS_LOCAL, exist_ok=True)
    for zp in (VQ1_ZIP, VQ2_ZIP):
        t0 = time.time()
        with zipfile.ZipFile(zp) as z:
            n = len(z.namelist())
            z.extractall(SESS_LOCAL)
        print(f"unzipped {n} frames from {os.path.basename(zp)} in {time.time()-t0:.0f} s")
else:
    print("frames already unzipped")

link = f"{CODE_LOCAL}/pilot/sessions"
if not os.path.islink(link) and not os.path.isdir(link):
    os.symlink(SESS_LOCAL, link)

t0 = time.time()
r = subprocess.run([sys.executable, f"{CODE_LOCAL}/pilot/perception/packcrops.py",
                    "--mode", "pack", "--out", LOCAL_CACHE, "--labels", LABELS_LOCAL],
                   capture_output=True, text=True)
print(r.stdout[-2000:] or r.stderr[-2000:])
assert r.returncode == 0, "pack failed -- read the output above"
meta_ = json.load(open(f"{LOCAL_CACHE}/meta.json"))
print(f"cache rebuilt in {time.time()-t0:.0f} s")
assert meta_["missing_frames"] == 0, \
    f"{meta_['missing_frames']} frames unreadable -- vq2_frames.zip is incomplete"
print(json.dumps(meta_, indent=1))

## 4. Import gatenet + packcrops + gatenet_rotaug, point at the cache

`G.LABELS` is set to the merged file BEFORE any attach so the fingerprint check hashes it.
Here we only load the cache (`P.attach`) for the sanity view and the split; the ROTATION
wrapper is installed in the train cell, so this cell is identical to the v3 baseline's.

In [ ]:
sys.path.insert(0, f"{CODE_LOCAL}/pilot/perception")
import gatenet as G
import packcrops as P
import gatenet_rotaug as R
import numpy as np

G.LABELS = LABELS_LOCAL
ANGLE = R.SAFE_ANGLE_DEG
print(f"safe roll angle: +/-{ANGLE:.2f} deg  (TILE_SCALE {P.TILE_SCALE} / MARGIN {G.MARGIN})")

items, tiles, meta = P.attach(LOCAL_CACHE)      # cached load_index/make_loaders (baseline)
print(f"instances: {len(items)}   tile {meta['tile_res']} scale {meta['tile_scale']}   "
      f"eval_exact={meta['eval_exact']}")

src_ = np.array([d['src'] for d in items])
rate_ = np.array([d['body_rate'] for d in items])
print(f"src: auto {int((src_=='auto').sum())} / hand {int((src_=='hand').sum())};   "
      f"body_rate>1 rad/s: {int((rate_>1).sum())} ({100*(rate_>1).mean():.1f}%)")

tr_items, va_items = G.split_index(items, "block")
print(f"block split: train {len(tr_items)} / val {len(va_items)}   "
      f"(hand in val: {sum(1 for d in va_items if d['src']=='hand')})")

### 4b. Sanity: the ROLLED tiles are pictures of gates, in the right frame

The check that catches a winding or roll-sign bug that a loss curve cannot. Each tile is a
train sample straight out of `RotAugCrops` -- so it is ROLLED by up to +/-23 deg. Green = the
target corners the net will be trained on, yellow dot = corner 0 (the min(u+v) winding
corner). If the roll is correct the green quad sits on the (rolled) aperture and the yellow
dot stays on the same physical corner picked by `canon`. The last row samples HAND-labelled
instances. If a quad drifts off an aperture or the yellow dot wanders, STOP.

In [ ]:
import cv2
from matplotlib import pyplot as plt

base_tr = P.CachedGateCrops(tr_items, LOCAL_CACHE, meta, train=True, hflip=False)
aug = R.RotAugCrops(base_tr, ANGLE)
hand_idx = [k for k, d in enumerate(tr_items) if d['src'] == 'hand']
sel = list(np.linspace(0, len(aug) - 1, 8).astype(int))
sel += list(np.array(hand_idx)[np.linspace(0, len(hand_idx) - 1,
                                           min(4, len(hand_idx))).astype(int)]) \
       if hand_idx else []
cols_ = 4
rows_ = (len(sel) + cols_ - 1) // cols_
fig, ax = plt.subplots(rows_, cols_, figsize=(16, 4 * rows_))
for a in np.ravel(ax):
    a.axis("off")
for a, k in zip(np.ravel(ax), sel):
    x, y, g, _ = aug[int(k)]
    img = ((x * 0.25 + 0.45) * 255).clamp(0, 255).byte().numpy().transpose(1, 2, 0)
    img = np.ascontiguousarray(img[:, :, ::-1])          # BGR -> RGB
    q = (y.numpy().reshape(4, 2) + 1.0) * (G.RES / 2.0)
    cv2.polylines(img, [q.astype(np.int32).reshape(-1, 1, 2)], True, (0, 255, 0), 1)
    cv2.circle(img, tuple(q[0].astype(int)), 4, (255, 255, 0), -1)
    d = tr_items[int(k)]
    a.imshow(img)
    a.set_title(f"{d['size_px']:.0f} px {d['src']}" + (" CLIP" if d['clipped'] else ""))
plt.tight_layout(); plt.show()

## 5. Train -- baseline hyperparameters, roll augmentation installed

`R.attach_rotaug` re-points `gatenet.load_index/make_loaders` at the cache AND wraps ONLY the
train loader in `RotAugCrops`; the val loader stays the plain `CachedGateCrops`. weight-mode
is `none` (the colab-v3-nw baseline). Set `RESUME = True` and re-run after a disconnect.

In [ ]:
import types, threading

TAG = "colab-v3-rot"
SPLIT = "block"
EPOCHS = 200
RESUME = False           # flip to True after a disconnect and re-run this cell
BATCH = 256
WORKERS = min(8, os.cpu_count() or 2)
MAX_HOURS = 3.0
WEIGHT_MODE = "none"     # colab-v3-nw baseline: NO rate weighting. Roll is the ONE change.
LR = 3e-4 * (BATCH / 64)  # linear LR scaling with batch, as in the v3 baseline

RUNS_LOCAL = "/content/gatenet_runs"
os.makedirs(f"{RUNS_LOCAL}/{TAG}", exist_ok=True)
os.makedirs(f"{RUNS_DRIVE}/{TAG}", exist_ok=True)
G.RUNS = RUNS_LOCAL
G.REPORT = "/content/TRAINING_colab_v3_rot.md"

# install the roll augmentation: train loader wrapped, val loader untouched
items, tiles, meta = R.attach_rotaug(LOCAL_CACHE, max_angle_deg=ANGLE)

_stop = threading.Event()
def _mirror(every=300):
    while not _stop.wait(every):
        try:
            for n in ("best.pt", "last.pt", "log.csv"):
                s = f"{RUNS_LOCAL}/{TAG}/{n}"
                if os.path.exists(s):
                    shutil.copyfile(s, f"{RUNS_DRIVE}/{TAG}/{n}")
        except Exception as e:
            print("mirror failed (training continues):", e)
threading.Thread(target=_mirror, daemon=True).start()

if RESUME:
    for n in ("best.pt", "last.pt", "log.csv"):
        s = f"{RUNS_DRIVE}/{TAG}/{n}"
        if os.path.exists(s) and not os.path.exists(f"{RUNS_LOCAL}/{TAG}/{n}"):
            shutil.copyfile(s, f"{RUNS_LOCAL}/{TAG}/{n}")

args = types.SimpleNamespace(
    mode="train", split=SPLIT, tag=TAG, epochs=EPOCHS, batch=BATCH, lr=LR,
    width=1.0, workers=WORKERS, max_hours=MAX_HOURS, resume=RESUME, hflip=False,
    ckpt="best", cpu=False, res=G.RES, seed=0, weight_mode=WEIGHT_MODE)

torch.manual_seed(args.seed); np.random.seed(args.seed)
cv2.setNumThreads(0); torch.backends.cudnn.benchmark = True
print(f"batch {BATCH}  lr {LR:.2e}  workers {WORKERS}  roll +/-{ANGLE:.2f} deg  "
      f"weight-mode {WEIGHT_MODE}  -> {RUNS_LOCAL}, mirrored to Drive")

t0 = time.time()
rows = G.train(args)
_stop.set()
for n in ("best.pt", "last.pt", "log.csv"):
    s = f"{RUNS_LOCAL}/{TAG}/{n}"
    if os.path.exists(s):
        shutil.copyfile(s, f"{RUNS_DRIVE}/{TAG}/{n}")
if os.path.exists(G.REPORT):
    shutil.copyfile(G.REPORT, f"{DRIVE}/TRAINING_colab_v3_rot.md")
print(f"\nwall clock {(time.time()-t0)/60:.1f} min; checkpoints mirrored to Drive")

## 6. The new model's report (per-rate / per-src rows)

Re-run the eval from the best checkpoint on the untouched val loader. Do NOT compare against
the old 0.89 px headline -- the label file changed. The decision is the next cell.

In [ ]:
from torch.utils.data import DataLoader

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
m = G.GateNet(1.0).to(dev)
ck = torch.load(f"{RUNS_DRIVE}/{TAG}/best.pt", map_location=dev, weights_only=False)
m.load_state_dict(ck["model"]); m.eval()
print(f"best.pt: epoch {ck['epoch']}, best val corner med {ck['best']:.3f} px, "
      f"{ck['nparam']:,} params")

va = DataLoader(P.CachedGateCrops(va_items, LOCAL_CACHE, meta, train=False),
                batch_size=256, shuffle=False, num_workers=WORKERS)
res_new = G.evaluate(m, va, dev, va_items)
rows_new = G.breakdown(res_new, va_items)
print()
print(G.fmt_rows(rows_new))

## 7. CROSS-EVAL and the frozen accept rule

The decision cell. **`colab-v3-rot` (new) vs `colab-v3-nw` (production)** on the IDENTICAL
val split. Both were trained on `labels_merged_v3.json`, so they share the block boundaries
and the val set EXACTLY -- this comparison has **no home-field leakage**, so the `va_items`
table below is the decision (unlike vs `block/best.pt`, which used the v1-era split; that one
is only fair on the clean subset, shown second).

**Frozen accept rule (decided before the numbers exist):** the whole point of roll augmentation
is robustness, so deploy `colab-v3-rot` only if
* the **ALL** row improves or holds within noise (new - old <= +0.1 px), AND
* the **60-120 px** row and the **clipped** row each improve or hold (<= +0.1 px).

Otherwise production stays `colab-v3-nw`. The per-body_rate rows are printed too: the
**>1 rad/s** row is where roll diversity should pay off, and it separates any label-aliasing
error from net error.

In [ ]:
# score NEW and the NW baseline on the SAME cached val loader
NEW = f"{RUNS_DRIVE}/{TAG}/best.pt"

def _score(ckpt_path):
    mm = G.GateNet(1.0).to(dev)
    mm.load_state_dict(torch.load(ckpt_path, map_location=dev, weights_only=False)["model"])
    mm.eval()
    ld = DataLoader(P.CachedGateCrops(va_items, LOCAL_CACHE, meta, train=False),
                    batch_size=256, shuffle=False, num_workers=WORKERS)
    return {r["group"]: r for r in G.breakdown(G.evaluate(mm, ld, dev, va_items), va_items)}

rows_nw = _score(NW_BEST)
rows_rt = {r["group"]: r for r in rows_new}   # from cell 6

groups = ["ALL", "size 15-30 px", "size 30-60 px", "size 60-120 px", "size >=120 px",
          "clipped", "unclipped", "body_rate 0-0.5 rad/s", "body_rate 0.5-1 rad/s",
          "body_rate >1 rad/s", "src hand (VQ2)"]
print("### colab-v3-rot (NEW) vs colab-v3-nw (production), SAME val split -- corner med px")
print("| group | n | nw | rot | delta |")
print("|---|---:|---:|---:|---:|")
for gname in groups:
    o, n = rows_nw.get(gname), rows_rt.get(gname)
    if not o or not n:
        continue
    print(f"| {gname} | {n['n']} | {o['corner_med']:.2f} | {n['corner_med']:.2f} | "
          f"{n['corner_med']-o['corner_med']:+.2f} |")

d_all = rows_rt["ALL"]["corner_med"] - rows_nw["ALL"]["corner_med"]
d_60 = rows_rt["size 60-120 px"]["corner_med"] - rows_nw["size 60-120 px"]["corner_med"]
d_clip = rows_rt["clipped"]["corner_med"] - rows_nw["clipped"]["corner_med"]
accept = (d_all <= 0.1) and (d_60 <= 0.1) and (d_clip <= 0.1)
print(f"\nALL {d_all:+.2f}  |  60-120px {d_60:+.2f}  |  clipped {d_clip:+.2f}   "
      f"(accept needs all <= +0.10 px)")
print("=> ACCEPT: colab-v3-rot becomes production" if accept else
      "=> REJECT: production stays colab-v3-nw")

### 7b. Clean-subset cross-check (the OLD production too, no leakage anywhere)

`val_clean_keys.json` is the frame set held out under BOTH the v1-era split (the OLD
`block/best.pt`) and the v3 split, so all three models are graded off their own training data.
Cached items carry `session`+`ordinal` but not the filename, so the `<session>/<file>` key is
reconstructed from the label file's sorted keys exactly as `gatenet.load_index` builds the
ordinal (verified on the laptop against the path-derived key: 0 mismatches).

In [ ]:
from collections import defaultdict

clean = set(json.load(open(f"{CODE_LOCAL}/pilot/perception/val_clean_keys.json"))["clean"])
raw = json.load(open(LABELS_LOCAL))
sess_files = defaultdict(list)
for kk in sorted(raw):
    s_, f_ = kk.split("/", 1)
    sess_files[s_].append(f_)
def _ckey(it):
    return f"{it['session']}/{sess_files[it['session']][it['ordinal']]}"

sub = [it for it in va_items if _ckey(it) in clean]
print(f"clean subset: {len(sub)} of {len(va_items)} val instances")

ld = DataLoader(P.CachedGateCrops(sub, LOCAL_CACHE, meta, train=False),
                batch_size=256, shuffle=False, num_workers=WORKERS)
ckpts = [("colab-v3-nw", NW_BEST), ("colab-v3-rot", NEW)]
if os.path.isfile(OLD_BEST):
    ckpts.insert(0, ("OLD block", OLD_BEST))
per = {}
for name, path in ckpts:
    mm = G.GateNet(1.0).to(dev)
    mm.load_state_dict(torch.load(path, map_location=dev, weights_only=False)["model"])
    mm.eval()
    per[name] = G.evaluate(mm, ld, dev, sub)["corner"].mean(1)
    del mm

size = np.array([it["size_px"] for it in sub])
clip = np.array([bool(it.get("clipped")) for it in sub])
rate = np.array([float(it.get("body_rate", 0.0)) for it in sub])
def _row(mask, label):
    if mask.sum() == 0:
        return
    cells = "  ".join(f"{np.median(per[n][mask]):6.2f}/{np.percentile(per[n][mask],90):6.2f}"
                      for n, _ in ckpts)
    print(f"  {label:20s} n={int(mask.sum()):4d}   {cells}")
print(" " * 30 + "   ".join(f"{n:>13s}" for n, _ in ckpts) + "   (med/p90 px)")
_row(np.ones(len(sub), bool), "ALL (clean)")
for lo, hi, lab in [(0,15,"size 0-15"),(15,30,"size 15-30"),(30,60,"size 30-60"),
                    (60,120,"size 60-120"),(120,1e9,"size >=120")]:
    _row((size>=lo)&(size<hi), lab)
_row(clip, "clipped")
for lo, hi, lab in [(0,0.5,"rate 0-0.5"),(0.5,1,"rate 0.5-1"),(1,1e9,"rate >1")]:
    _row((rate>=lo)&(rate<hi), lab)

## 8. Bring the result home

Copy `gatenet_runs/colab-v3-rot/` back into `pilot/perception/gatenet_runs/` on the laptop
and paste the decision table + the clean-subset table into `TRAINING.md` under a heading
naming the GPU from cell 1. If ACCEPT, `colab-v3-rot/best.pt` is the new production; if REJECT,
the per-rate rows are the diagnosis (did the >1 rad/s row move at all?).

In [ ]:
print("on Drive:", RUNS_DRIVE + "/" + TAG)
for f in sorted(os.listdir(f"{RUNS_DRIVE}/{TAG}")):
    print(f"  {f}  {os.path.getsize(f'{RUNS_DRIVE}/{TAG}/{f}')/1e6:.1f} MB")
import csv as _csv
rows_ = list(_csv.DictReader(open(f"{RUNS_DRIVE}/{TAG}/log.csv")))
secs = [float(r["secs"]) for r in rows_]
if secs:
    print(f"\nper-epoch times (s): n={len(secs)}  median {sorted(secs)[len(secs)//2]:.1f}  "
          f"min {min(secs):.1f}  max {max(secs):.1f}")